# 📥 Lesson 02: Loading

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HosseinHeydari2004/RAG-Tutorial-From-Zero-to-Advanced/blob/main/notebooks/02_loading.ipynb)

> **Before a model can use your document, Python has to be able to read it.**

---

## 🎯 Goal
By the end of this lesson you can load a PDF into Python as a list of `Document` objects, each holding the text and the metadata of one page.

## ℹ️ Lesson info

| | |
|---|---|
| ⏱ **Time** | ~15 minutes |
| 📚 **Prerequisites** | [Lesson 01](01_core_concepts.ipynb) |
| 🔑 **API key** | Not needed |
| 🛠 **Libraries** | `pypdf`, `langchain-core` |


In [ ]:
!pip install -q -U pypdf langchain-core

In [1]:
from pathlib import Path
import urllib.request

PDF_URL = "https://raw.githubusercontent.com/HosseinHeydari2004/RAG-Tutorial-From-Zero-to-Advanced/main/data/pdf1.pdf"

pdf_path = next((p for p in [Path("data/pdf1.pdf"), Path("../data/pdf1.pdf")] if p.exists()), None)

if pdf_path is None:
    pdf_path = Path("data/pdf1.pdf")
    pdf_path.parent.mkdir(exist_ok=True)
    urllib.request.urlretrieve(PDF_URL, pdf_path)

print("using:", pdf_path)

using: ..\data\pdf1.pdf


## 📦 What is a `Document`?

A RAG system doesn't work with "files". It works with **Documents**: a small object holding two things:

| Field | What it is | Example |
|---|---|---|
| `page_content` | the text | `"Employee Handbook ..."` |
| `metadata` | information *about* the text | `{"source": "pdf1.pdf", "page": 0}` |

Why keep the metadata? Later we will use it to **filter** results and to **cite** where an answer came from ("see page 3").

A PDF has pages, so we create **one Document per page**.

In [4]:
from pypdf import PdfReader
from langchain_core.documents import Document
reader = PdfReader(pdf_path)
data = []
for page_number, page in enumerate(reader.pages):
    data.append(
        Document(
            page_content=page.extract_text(),
            metadata={"source":pdf_path.name, "page":page_number}
        ),
    )
print("pages loaded:", len(data))

incorrect startxref pointer(1)
parsing for Object Streams


pages loaded: 1


### 👀 Look inside

`data` is a plain Python list. Each item is a `Document`. Let's inspect the first one.

In [5]:
print(data[0].metadata)
print()
print(data[0].page_content[:300])

{'source': 'pdf1.pdf', 'page': 0}

Employee Handbook
Non-Disclosure Agreement (NDA) Policy
Employees must protect confidential information belonging to the company, its clients, and partners.
This includes, but is not limited to, product roadmaps, customer data, internal communications,
proprietary algorithms, financial information, 


## 🔍 Extraction is not perfect

A PDF is a *drawing* of text, not a text file. A page just says "put these shapes at these positions", and the library has to guess which characters they are and where the lines break.

So the extracted text often has small defects. Let's hunt for two of them in our page:

In [6]:
text = data[0].page_content

# 1) Strange characters
i = text.find("Full")
print(repr(text[i:i + 12]))
print("non-ASCII characters:", sorted({c for c in text if ord(c) > 127}))

# 2) Line breaks in the middle of sentences
print("newlines:", text.count("\n"))
print(repr(text[100:260]))

'Full■time em'
non-ASCII characters: ['–', '■']
newlines: 25
'ion belonging to the company, its clients, and partners.\nThis includes, but is not limited to, product roadmaps, customer data, internal communications,\nproprie'


## 📝 A different format, the same shape

Whatever the source (PDF, text file, web page, Word file), the goal of *loading* is always the same: end up with a list of `Document` objects. Everything after this lesson only has to understand that one shape.

A plain text file is the easiest case, because there is nothing to extract. We just read it. We create a small one first, so the lesson needs no extra download:

In [7]:
import tempfile

txt_path = Path(tempfile.gettempdir()) / "remote_policy.txt"
txt_path.write_text(
    "Remote Work Policy\nEmployees may work remotely up to 3 days per week.\n",
    encoding="utf-8",
)

txt_document = Document(
    page_content=txt_path.read_text(encoding="utf-8"),
    metadata={"source": txt_path.name, "page": 0},
)

all_documents = data + [txt_document]

for doc in all_documents:
    print(doc.metadata, "->", len(doc.page_content), "characters")

{'source': 'pdf1.pdf', 'page': 0} -> 1943 characters
{'source': 'remote_policy.txt', 'page': 0} -> 70 characters


## 📌 Summary

- **Loading** means turning a file into a list of `Document` objects, so the rest of the pipeline only needs to understand one shape.
- A `Document` has `page_content` (the text) and `metadata` (information *about* the text, such as the source file and page number).
- For a PDF we made **one Document per page**. Page numbers start at **0**.
- Extraction is **not perfect**: we found a `■` instead of a hyphen in "Full-time", and line breaks in the middle of sentences. Always look at the text you loaded.
- A plain text file needs no extraction. Different formats, same shape.

## ✍️ Exercise (10 minutes)

Write a function `load_text_file(path)` that takes the path of a `.txt` file and returns a `Document` with `page_content` and `metadata` (`source` and `page`).

1. Test it on `remote_policy.txt` from this lesson.
2. Create a second `.txt` file with any text, load it with your function, and add it to `all_documents`.

**Success:** printing `all_documents` shows three documents, each with its own `source`.

💡 Solution: `solutions/02_loading.ipynb` (try on your own first)

## 🔥 Challenge (optional)

Find a PDF with **several pages** (any report or paper you have) and load it with the same code.

1. How many `Document`s did you get? Does it match the number of pages?
2. Print the first 200 characters of two different pages. Do you see headers, footers or page numbers mixed into the text?

Think about it: if a footer like "Confidential - Page 7" appears on *every* page, what would it do to a search later? We fix this kind of noise in the Cleaning lesson.

## 📖 Next

We loaded the text, but a PDF is more than plain text: it has titles, lists and tables. In the next lesson we try to keep that **structure**.

**➡️ Next lesson:** [03 Parsing](03_parsing.ipynb)